In [ ]:
import os
import pandas as pd
import torch
import math
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
BASE_MODEL_PATH = "/kaggle/input/akkad-byt5-model-v3-finetune-3epochs"
BATCH_SIZE = 4
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⚙️ Environnement : {DEVICE}")

# --- 2. LOCALISATION INTELLIGENTE DU MODÈLE ---
model_path = BASE_MODEL_PATH
for root, dirs, files in os.walk(BASE_MODEL_PATH):
    if "config.json" in files:
        model_path = root
        break

print(f"📂 Modèle détecté dans : {model_path}")

# --- 3. CHARGEMENT DES MODÈLES ---

# A. Modèle de Traduction (ByT5)
print("⏳ Chargement du Traducteur (ByT5)...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
model.eval()

# ==============================================================================
# 3. LE JUGE HEURISTIQUE (ANTI-RÉPÉTITION & OFFLINE)
# ==============================================================================
class PythonHeuristicJudge:
    def __init__(self):
        pass 

    def score(self, sentence):
        """
        Calcule une pénalité. 
        Plus le score est HAUT, plus la phrase est MAUVAISE.
        """
        clean_sent = sentence.strip()
        words = clean_sent.split()
        
        # 1. Hallucinations vides (toujours à rejeter)
        if not words: return 9999.0
        
        # 2. Ratio de Diversité (Le Juge Anti-Boucle)
        # Ex: "King King King" -> Ratio 0.33 -> Pénalité énorme
        # Ex: Phrase de 70 mots variée -> Ratio 0.8 -> Pénalité faible
        unique_words = set([w.lower() for w in words])
        diversity_ratio = len(unique_words) / len(words) if len(words) > 0 else 0
        
        # Pénalité explosive si le vocabulaire est pauvre
        penalty_repetition = (1.0 - diversity_ratio) * 500.0 
        
        # 3. Pénalité de Longueur (Ajustée pour tes phrases de 70 mots)
        # On ne commence à punir qu'à partir de 100 mots.
        penalty_length = 0
        if len(words) > 100:
            penalty_length = (len(words) - 100) * 10 
        
        # 4. Pénalité Mots Interdits (Boucles connues)
        penalty_keywords = 0
        text_lower = clean_sent.lower()
        if "of the land of the land" in text_lower: penalty_keywords += 200
        # On punit les triple répétitions "King of the king of the king"
        if "king of the king of the king" in text_lower: penalty_keywords += 200

        return penalty_repetition + penalty_length + penalty_keywords

    def pick_best(self, candidates):
        scored_candidates = []
        for cand in candidates:
            s = self.score(cand)
            scored_candidates.append((s, cand))
        
        # On trie : le plus petit score (la phrase la plus propre) gagne
        scored_candidates.sort(key=lambda x: x[0])
        
        # Le premier de la liste est le meilleur
        return scored_candidates[0][1]

# Activation du Juge
judge = PythonHeuristicJudge()
print("🛡️ Juge Heuristique activé (Tolérance Longueur: 100 mots).")
# --- 4. CHARGEMENT DES DONNÉES DE TEST ---
comp_dir = "/kaggle/input/deep-past-initiative-machine-translation"
if not os.path.exists(comp_dir):
    input_dirs = [d for d in os.listdir("/kaggle/input") if "deep-past" in d or "translation" in d]
    if input_dirs:
        comp_dir = os.path.join("/kaggle/input", input_dirs[0])

test_file = os.path.join(comp_dir, "test.csv")
print(f"📄 Chargement du fichier test : {test_file}")
test_df = pd.read_csv(test_file)

# --- 5. MOTEUR DE TRADUCTION AVEC RERANKING ---
def translate_batch_with_reranking(texts, batch_size=4):
    final_translations = []
    # Préfixe (Si ton modèle a été entraîné avec)
    inputs_text = ["translate Akkadian to English: " + str(t).strip() for t in texts]
    
    # Nombre de candidats par phrase
    N_CANDIDATES = 10
    
    for i in tqdm(range(0, len(inputs_text), batch_size), desc="Traduction + Vote"):
        batch = inputs_text[i : i + batch_size]
        
        inputs = tokenizer(
            batch, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=MAX_LENGTH
        ).to(DEVICE)
        
        with torch.no_grad():
            # On génère 5 versions pour CHAQUE phrase du batch
            generated_ids = model.generate(
                **inputs,
                max_length=512,
                num_beams=N_CANDIDATES,             # Beam Search large
                num_return_sequences=N_CANDIDATES,  # On veut voir les 5 options
                repetition_penalty=1.2,  # ⚠️ PÉNALITÉ FORTE pour éviter les boucles à la source
                early_stopping=True
            )
        
        # Décodage de tous les candidats
        decoded_candidates = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        
        # decoded_candidates est une liste plate [P1_v1, P1_v2, ..., P1_v5, P2_v1, ...]
        # Il faut les regrouper par paquet de 5 pour faire voter le juge
        
        for j in range(0, len(decoded_candidates), N_CANDIDATES):
            # On prend les 5 versions de la phrase actuelle
            group_of_5 = decoded_candidates[j : j + N_CANDIDATES]
            
            # Le juge choisit la meilleure (celle qui ne bugge pas)
            best_sent = judge.pick_best(group_of_5)
            final_translations.append(best_sent)
            
    return final_translations

# --- 6. EXÉCUTION ---
print(f"🔥 Démarrage de la traduction (Best-of-5) de {len(test_df)} phrases...")
# Attention : batch_size réduit car on génère 5x plus de texte en mémoire
test_df['translation'] = translate_batch_with_reranking(test_df['transliteration'].tolist(), batch_size=4)

# --- 7. SAUVEGARDE ---
submission = test_df[['id', 'translation']]
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("\n" + "="*40)
print("✅ submission.csv GÉNÉRÉ AVEC SUCCÈS (Nettoyé par GPT-2) !")
print("👉 Tu peux cliquer sur 'Submit' à droite.")
print("="*40)

#2323